# 0d · Self-referential (stance) vs descriptive sentiment — decomposing the reflection

**Motivation (from the discussion).** The whole-reflection sentiment blends two kinds of sentence: **descriptive** ('Randall seems devastated' — the *character's* state) and **stance** ('I feel bad for him' — the *viewer's* feeling). Whole-text sentiment tracks the character's `positive emotion` and misses `like` (~0.07). **Hypothesis:** that null is a *dilution* — split the reflection, score each subset, and stance-sentiment should recover `like` while descriptive-sentiment tracks `positive emotion`. If so, verbalized feeling IS recoverable; it's just concentrated in the ~20% stance sentences and drowned out by the descriptive majority.

Data: ~3,700 sentences, 53% descriptive / 47% first-person. **Scaffold — run 0d.3 locally (transformer).**

## 0d.1 · Load + sentence-split

In [1]:
# 0d.1 · Load reflections and split into sentences.
import pandas as pd, numpy as np, re
sc = pd.read_csv("results/scored/00__reflection_sentiment.csv")[["Participant","Run","Character","Raw_Text"]].copy()
sc = sc.dropna(subset=["Raw_Text"]); sc["Character"] = sc["Character"].str.lower().str.strip()
def split_sentences(t):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", str(t)) if s.strip()]
sent = sc.assign(sentence=sc["Raw_Text"].map(split_sentences)).explode("sentence").reset_index(drop=True)
sent = sent[sent["sentence"].str.split().str.len() >= 2]
print(f"{len(sc)} reflections -> {len(sent)} sentences ({len(sent)/len(sc):.1f} per reflection)")


1272 reflections -> 3606 sentences (2.8 per reflection)


## 0d.2 · Classify stance vs descriptive

Rule-based first pass; the hard case is the **'I feel like' hedge** (= 'I think', NOT a feeling). An LLM classifier (hook at the bottom of the cell) is the recommended upgrade.

In [2]:
# 0d.2 · Classify each sentence: STANCE (viewer's feeling toward character) vs DESCRIPTIVE (character's state).
# RULE-BASED first pass -- deliberately conservative. KEY PROBLEM: "I feel like / I think / I'm assuming" are
# COGNITIVE HEDGES, not feelings, so they must NOT count as stance. An LLM classifier (hook below) is the
# recommended upgrade; the rule-based version is for prototyping / a sanity floor.
import re
FIRST   = re.compile(r"\b(i|i'm|im|i've|ive|i'd|id|me|my|myself)\b", re.I)
HEDGE   = re.compile(r"\b(i\s+feel\s+like|feels?\s+like|seems?\s+like|i\s+think|i'?m\s+assuming|i\s+guess|i\s+(don'?t|do\s+not)\s+know|i'?m\s+not\s+sure|i\s+mean|i\s+wonder)\b", re.I)
# genuine stance: first-person subject + affective/evaluative predicate ABOUT the character
STANCE  = re.compile(r"\bi\b[^.?!]{0,25}\b(feel\s+(bad|sorry|for)|felt|like[sd]?\b|love[sd]?|hate[sd]?|dislike|enjoy|relat(e|ed|able)|empathiz|sympath|admire|respect|pity|root(ing)?\s+for|care\s+about|connect(ed)?|found\s+(him|her|them))\b", re.I)
def classify(s):
    if STANCE.search(s) and not (HEDGE.search(s) and not re.search(r"\bi\s+(like|love|hate|feel\s+bad|feel\s+sorry|relate|empathiz|admire|respect|pity|care)\b", s, re.I)):
        return "stance"
    if FIRST.search(s) and not HEDGE.search(s) and re.search(r"\b(feel|felt|like|love|hate|relate|empath|admire|respect|pity|care|enjoy)\w*", s, re.I):
        return "stance"
    return "descriptive"
sent["kind"] = sent["sentence"].map(classify)
print(sent["kind"].value_counts(), "\n")
print("STANCE examples:");    [print("  +", x[:88]) for x in sent[sent.kind=="stance"]["sentence"].head(6)]
print("\nDESCRIPTIVE examples:"); [print("  -", x[:88]) for x in sent[sent.kind=="descriptive"]["sentence"].head(6)]

# ---- LOCAL zero-shot classifier (recommended upgrade; NO API -- same roberta-large-mnli as Step 0) ----
# Rule-based above misfires on hedges ('I feel like'). This is a LOCAL, reproducible improvement. It is
# per-sentence and slower, so run once and cache. Uncomment to use it instead of the rule-based `classify`.
# from transformers import pipeline
# _zs = pipeline('zero-shot-classification', model='roberta-large-mnli')   # local model, no network at inference
# _LBL = {'stance':      "the speaker's own feeling or attitude toward the character",
#         'descriptive': "a description of the character's state or behavior"}
# def classify_zeroshot(s):
#     out = _zs(s, candidate_labels=list(_LBL.values()))
#     return 'stance' if out['labels'][0] == _LBL['stance'] else 'descriptive'
# sent['kind'] = sent['sentence'].map(classify_zeroshot)   # overwrites the rule-based labels; cache to disk
#
# (For higher accuracy still, a local instruction model e.g. google/flan-t5-large with a STANCE/DESCRIPTIVE
#  prompt also runs fully locally -- no API. Both keep the pipeline reproducible.)


kind
descriptive    3275
stance          331
Name: count, dtype: int64 

STANCE examples:
  + It didn't change that much of my thoughts about him but it does show me a different side
  + Um, I'm kind of, I'm starting to like him more.
  + Randall, I wouldn't say my thoughts on him have really changed, but I now know he has a 
  + But also I thought it was really interesting how still thebhis sister's home even in the
  + I can't quite tell, but he was on a movie set and then he seemed to have like a break wh
  + My thoughts about Kate are that I feel bad for her.

DESCRIPTIVE examples:
  - I don't really feel like I've learned that much about Jack except that he has I'm assumi
  - Randall seems to be having a very emotional moment.
  - Um to have I guess step parents.
  - So, he never knew or, I don't know, but was not raised by his birth parents, um, seems r
  - Kevin seems in one scene very emotional um and angry and then the other he seemed very c
  - Kate seems very upset.


[None, None, None, None, None, None]

## 0d.3 · Score each subset (RUN LOCALLY)

In [3]:
# 0d.3 · Score STANCE vs DESCRIPTIVE text separately, per (participant, run, character). RUN LOCALLY (transformer).
# Rejoin each subset's sentences into one text per cell, then score with the Step-0 winner (Twitter-RoBERTa).
import numpy as np, pandas as pd, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
_TW = "cardiffnlp/twitter-roberta-base-sentiment-latest"
_tok = AutoTokenizer.from_pretrained(_TW); _mdl = AutoModelForSequenceClassification.from_pretrained(_TW).eval()
def _val(text):
    if not str(text).strip(): return np.nan
    p = softmax(_mdl(**_tok(str(text), return_tensors="pt", truncation=True, max_length=512)).logits.detach().numpy()[0])
    return float(p[2] - p[0])                      # pos - neg
rows = []
for (P, R, C), g in sent.groupby(["Participant","Run","Character"]):
    d = {"Participant": P, "Run": R, "Character": C}
    for kind in ["stance", "descriptive"]:
        joined = " ".join(g[g.kind == kind]["sentence"].tolist())
        d[f"val_{kind}"] = _val(joined); d[f"n_{kind}"] = int((g.kind == kind).sum())
    rows.append(d)
split = pd.DataFrame(rows)
os.makedirs("results/scored", exist_ok=True)
split.to_csv("results/scored/0d__split_sentiment.csv", index=False)
print("wrote results/scored/0d__split_sentiment.csv", split.shape)
print("cells with >=1 stance sentence:", int((split.n_stance>0).sum()), "/", len(split))


/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


wrote results/scored/0d__split_sentiment.csv (1271, 7)
cells with >=1 stance sentence: 275 / 1271


## 0d.4 · Validation — the decisive test

In [4]:
# 0d.4 · VALIDATION — the key test. Does STANCE-sentiment track LIKE, and DESCRIPTIVE track POSITIVE EMOTION?
# Group-level (same grid as Step 1). If stance-sentiment recovers 'like' where whole-text sentiment could not,
# that CONFIRMS the whole-text null on liking was a dilution by descriptive sentences -- verbalized feeling IS
# recoverable, just concentrated in the minority stance sentences.
import pandas as pd, numpy as np, re
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
from scipy.stats import spearmanr
split = pd.read_csv("results/scored/0d__split_sentiment.csv"); split["Character"] = split["Character"].str.lower().str.strip()
split["group"] = split["Participant"].map(lambda s:int(next(c for c in str(s) if c.isdigit())))
gt = pd.read_csv("results/step1/01__ground_truth_group_level.csv"); gt["Character"] = gt["Character"].str.lower().str.strip()
logo = LeaveOneGroupOut()
def cv(valcol, target):
    g = split.dropna(subset=[valcol]).groupby(["group","Character","Run"])[valcol].mean().reset_index()
    m = gt[["group","Character","Run",target]].merge(g, on=["group","Character","Run"]).dropna()
    if len(m) < 8: return np.nan, np.nan, len(m)
    r2 = cross_val_score(LinearRegression(), m[[valcol]].values, m[target].values, groups=m["group"].values, cv=logo, scoring="r2").mean()
    return round(r2,3), round(spearmanr(m[valcol], m[target])[0],3), len(m)
print("cv-R2 / rho (leave-one-group-out):")
for valcol in ["val_descriptive","val_stance"]:
    for target in ["positive emotion","like"]:
        r2,rho,n = cv(valcol, target)
        print(f"  {valcol:16s} -> {target:16s} : cv-R2={r2}  rho={rho}  (n={n})")
print("\nPREDICTION: val_descriptive -> positive emotion strong; val_stance -> like stronger than whole-text (0.07).")
print("Caveat: stance sentences are ~20% of text and sparse per cell, so val_stance is noisier -- interpret with the n.")


cv-R2 / rho (leave-one-group-out):
  val_descriptive  -> positive emotion : cv-R2=0.272  rho=0.553  (n=120)
  val_descriptive  -> like             : cv-R2=0.049  rho=0.392  (n=120)
  val_stance       -> positive emotion : cv-R2=0.133  rho=0.388  (n=104)
  val_stance       -> like             : cv-R2=-0.052  rho=0.218  (n=104)

PREDICTION: val_descriptive -> positive emotion strong; val_stance -> like stronger than whole-text (0.07).
Caveat: stance sentences are ~20% of text and sparse per cell, so val_stance is noisier -- interpret with the n.


## Reading it

- **val_descriptive -> positive emotion (high), val_stance -> like (higher than the 0.07 whole-text floor):** confirms the dissociation is about *sentence type*, and that verbalized feeling is recoverable. Strong result.
- **val_stance -> like stays ~0:** then liking really isn't verbalized even when people speak in the first person here (consistent with the 'thoughts about the character' prompt), and the whole-text null is not just dilution.
- Either outcome is informative and directly answers 'can transcribed speech capture the viewer's feeling.'
- **Upgrade path:** replace the rule-based classifier with the LLM hook; score all 6 Step-0 models, not just the winner.